# Evaluate.ipynb — optimization-pipeline performance evaluation

Compares the performance of the optimization modes produced by
`AugmentedLagrangian.ipynb`.

**2011-2012** — **seven** pipelines, evaluated together on the **same 616 data samples**.
A retry pipeline always ran its primary mode first, so the *same artefacts* let us score the
primary mode standalone **and** the full retry pipeline.  `sig_only [2]` and `sig_only [2] + sig_nu`
come from the **same root**, so they form a precise within-root primary-vs-retry comparison
(`sig_only [1]` is the separate retry-free root):

| # | pipeline | how it is scored | source root |
|---|---|---|---|
| 1 | **sig_nu** | primary-mode run only | `testing_2011_2012_signu_vs_sigonly` |
| 2 | **sig_nu + sig_only (retry)** | best of primary/retry; run-time = both solves | `testing_2011_2012_signu_vs_sigonly` |
| 3 | **sig_only [1]** | the genuinely retry-free root | `testing_2011_2012_signu_vs_sigonly` |
| 4 | **sig_only [2]** | primary-mode run only | `testing_2011_2022_sigonly` |
| 5 | **sig_only [2] + sig_nu (retry)** | best of primary/retry; run-time = both solves | `testing_2011_2022_sigonly` |
| 6 | **sig_only_LKbar** | primary-mode run only | `testing_2011_2012_sigonlyLU` |
| 7 | **sig_only_LKbar + sig_nu (retry)** | best of primary/retry; run-time = both solves | `testing_2011_2012_sigonlyLU` |

**2024-2025** — a single pipeline, **sig_only + sig_nu (retry)** (`testing_2024_2025`).

**How the artefacts are read**
* Each expiry sub-folder holds one `fit_results_<stamp>_<mode>.csv` per *mode-run* (the
  meta block at the top carries `optimize_mode`, `true_roughness_reduction_pct`,
  `run_time_sec`, `n_outer_iterations`, …), one `manifest_<expiry>.csv` (gives tenor `T`),
  and the `compare_summary_<stamp>_<primary_mode>_<expiry>_batchNN.png` whose filename
  reveals the folder's **primary** optimize-mode.
* A date is **retried** when its primary run scored `true_reduce < 95%` (`retry_threshold`
  in the script); the pipeline keeps the better of the two solves by `true_reduce`.
  We reconstruct this by grouping a folder's mode-runs by `start_date` and taking the
  run with the highest `true_reduce` as the *winner*.

**Outputs** (written to `evaluation/`):
* `eval_2011_2012_compare_1.svg` (+ `_compare_1_panels/`) — the explicit 7-way grid: roughness /
  run-time / true-reduction **vs data-point index** and **vs time-to-expiry**, `S_init` vs index & T,
  **mean true reduction by expiry (scatter)**, 95-100% true-reduction zoom-ins, **CDF (99-100% zoom
  + full)**, roughness-vs-true, and failure counts.  Each plot is also saved as its own panel image.
* `eval_2011_2012_compare_2.svg` (+ `_compare_2_panels/`) — per scatter metric, two facet sets:
  **Set A** one plain subplot per pipeline, **Set B** the entire dataset in faint grey behind each
  pipeline's coloured points.  Each metric also saved as its own panel.
* `eval_2011_2012_compare_3.svg` (+ `_compare_3_panels/`) — **box plots** of run-time, roughness
  and true reduction, one box per pipeline; each metric also a panel.
* `eval_2011_2012_stats_table.svg` — the per-pipeline stats table (true reduction, roughness,
  run-time, **iters-to-converge**) as its own image.  Also `eval_2011_2012_stats.tex` (LaTeX).
* `eval_2011_2012_convergence.svg` — from the run_logs: **feasibility track** (fraction of runs
  feasible vs outer iteration), **iters-to-converge** boxen, **fraction-feasible** boxen.
* `eval_2011_2012_main.svg`, `_extra.svg`, `_data.csv` — core/extra comparison + per-date data.
* `eval_2024_2025_{main,extra}.svg` + `eval_2024_2025_data.csv` — the single 2024-2025 pipeline.

Every csv carries per-date rows **and** per-pipeline summary rows, distinguished by `record_type`.

Run the whole thing with the single call at the bottom: **`evaluate_all()`**.


In [1]:
import glob, os, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # headless: figures go straight to disk
matplotlib.rcParams["savefig.dpi"] = 150   # resolution of rasterized scatter layers in the SVGs
import matplotlib.pyplot as plt

BASE_DIR  = "."                # root that holds the testing_* folders
EVAL_DIR  = "evaluation"       # where images + csvs are written
IMG_EXT   = "svg"              # vector output format for every figure
PNG_DPI   = 200                # also drop a crisp (high-dpi) png next to every vector figure
THRESHOLD = 95.0               # true-roughness-reduction %; == retry_threshold in the script
PALETTE   = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2"]


def _save(fig, path, **kw):
    # save the VECTOR figure and, alongside it, a crisp high-dpi PNG (same basename)
    fig.savefig(path, **kw)
    kw_png = {k: v for k, v in kw.items() if k != "dpi"}
    fig.savefig(os.path.splitext(path)[0] + ".png", dpi=PNG_DPI, **kw_png)

# short legend/axis labels; retry pipelines name the SPECIFIC retry mode (not a generic "+retry")
SHORT = {
    "sig_nu": "sig_nu",
    "sig_nu + sig_only (retry)": "sig_nu+sig_only",
    "sig_only": "sig_only",
    "sig_only (no retry)": "sig_only",
    "sig_only [1]": "sig_only[1]",
    "sig_only [2]": "sig_only[2]",
    "sig_only [2] + sig_nu (retry)": "sig_only[2]+sig_nu",
    "sig_only + sig_nu (retry)": "sig_only+sig_nu",
    "sig_only_LKbar": "sig_LKbar",
    "sig_only_LKbar + sig_nu (retry)": "sig_LKbar+sig_nu",
}
def _short(p):
    return SHORT.get(p, p.split(" ")[0])

def _sep(label):
    # title prefix: "<label>  —  " when a label is set, else nothing
    return (label + "  —  ") if label else ""

# single-mode (primary-only) pipelines get distinct non-circle markers so they stay
# distinguishable under overdraw regardless of plot order; retry (double) pipelines keep "o"
_SINGLE_MARKERS = ["x", "^", "s", "D", "v", "P", "*"]
def _markers(pipes):
    out, si = [], 0
    for p in pipes:
        if "retry" in str(p):
            out.append("o")
        else:
            out.append(_SINGLE_MARKERS[si % len(_SINGLE_MARKERS)]); si += 1
    return out

def _zorders(pipes):
    # draw order via zorder: default = legend/canonical order, BUT draw sig_nu AFTER (on top of)
    # sig_nu+sig_only.  Legend order is unaffected (we still scatter in canonical order).
    z = {p: i + 2 for i, p in enumerate(pipes)}
    if "sig_nu" in z and "sig_nu + sig_only (retry)" in z:
        z["sig_nu"] = z["sig_nu + sig_only (retry)"] + 0.5
    return [z[p] for p in pipes]

# Which pipeline(s) live under each testing root, keyed by the folder's PRIMARY optimize_mode
# (read off the compare_summary_*.png filename).  retry=True means a sub-threshold primary
# run triggered a second solve in another mode (the retry mode is detected per-date).
ROOT_CONFIG = {
    "testing_2011_2012_signu_vs_sigonly": {
        "label": "2011-2012  sig_nu+retry  vs  sig_only(no-retry)",
        "pipelines": {
            "sig_nu":   {"name": "sig_nu + sig_only (retry)", "retry": True},
            "sig_only": {"name": "sig_only (no retry)",        "retry": False},
        },
    },
    "testing_2011_2012_sigonlyLU": {
        "label": "2011-2012  sig_only_LKbar + sig_nu (retry)",
        "pipelines": {
            "sig_only_LKbar": {"name": "sig_only_LKbar + sig_nu (retry)", "retry": True},
        },
    },
    "testing_2024_2025": {
        "label": "2024-2025  sig_only + sig_nu (retry)",
        "pipelines": {
            "sig_only": {"name": "sig_only + sig_nu (retry)", "retry": True},
        },
    },
}

_MODE_RE = re.compile(r"(sig_nu|sig_only_LKbar|sig_only)_\d{8}_batch")


def _folder_primary_mode(folder):
    # primary optimize_mode of an expiry folder, from its compare_summary_*.png name
    for cs in glob.glob(os.path.join(folder, "compare_summary_*")):
        m = _MODE_RE.search(os.path.basename(cs))
        if m:
            return m.group(1)
    return None


def _read_meta(fp):
    # fast read of just the meta block (it sits at the TOP of every fit_results csv)
    df = pd.read_csv(fp, nrows=30)
    m = df[df["record_type"] == "meta"]
    return dict(zip(m["key"], m["value"]))


_LOG_ITER_RE = re.compile(r"J=([-\d.einf]+)\s*\|\s*max_viol=([\d.eE+-]+)\s*\|\s*feasible=(True|False)")
_MAX_TRACK = 100   # all runs hit the 100-outer cap; pad/truncate feasibility tracks to this


def _parse_one_log(fp):
    # extract per-outer-iteration trajectory metrics from one run_log_*.txt
    txt = open(fp, errors="ignore").read()
    hsd = re.search(r"start_date=(\d+)", txt)
    hmd = re.search(r"optimize_mode=(\S+)", txt)
    iters = _LOG_ITER_RE.findall(txt)
    if not (hsd and hmd and iters):
        return None
    Ji = re.search(r"J_initial=([-\d.]+)", txt)
    Jf = re.search(r"J_final=([-\d.]+)", txt)
    Js, viols, feas = [], [], []
    for j, v, fl in iters:
        try: Js.append(float(j))
        except Exception: Js.append(np.nan)
        try: viols.append(float(v))
        except Exception: viols.append(np.nan)
        feas.append(fl == "True")
    Js = np.asarray(Js, float); feas = np.asarray(feas, float)
    J0 = float(Ji.group(1)) if Ji else (Js[0] if len(Js) else np.nan)
    Jfin = float(Jf.group(1)) if Jf else (np.nanmin(Js) if len(Js) else np.nan)
    # ITERS TO CONVERGENCE := first outer iter where J reaches within 1% of its TOTAL improvement
    # (all runs hit the 100 cap, so n_outer is uninformative; this measures effective speed)
    if np.isfinite(J0) and np.isfinite(Jfin) and J0 != Jfin:
        hit = np.where(Js <= (J0 - 0.99 * (J0 - Jfin)))[0]
        itc = int(hit[0]) + 1 if len(hit) else len(Js)
    else:
        itc = len(Js)
    track = np.full(_MAX_TRACK, np.nan)
    track[:min(len(feas), _MAX_TRACK)] = feas[:_MAX_TRACK]
    return dict(start_date=int(hsd.group(1)), mode=hmd.group(1),
                iters_to_conv=int(itc),
                first_feas=(int(np.argmax(feas)) + 1 if feas.any() else np.nan),
                frac_feas=float(feas.mean()) if len(feas) else np.nan,
                final_viol=float(viols[-1]) if viols else np.nan,
                feas_track=track)


def _parse_run_logs(folder):
    # {(start_date, optimize_mode): trajectory metrics} for every run_log in a folder
    out = {}
    for fp in glob.glob(os.path.join(folder, "run_log_*.txt")):
        try:
            d = _parse_one_log(fp)
            if d:
                out[(d["start_date"], d["mode"])] = d
        except Exception:
            pass
    return out


def collect_runs(root_path, root_key):
    # one row per mode-run (per fit_results csv) under a testing root
    rows = []
    for folder in sorted(glob.glob(os.path.join(root_path, "*"))):
        if not os.path.isdir(folder):
            continue
        prim = _folder_primary_mode(folder)
        fits = glob.glob(os.path.join(folder, "fit_results_*.csv"))
        if not fits:
            continue
        # tenor T (years) per start_date comes from the folder manifest, not the meta block
        Tmap = {}
        for man in glob.glob(os.path.join(folder, "manifest_*.csv")):
            try:
                md_ = pd.read_csv(man)
                Tmap.update(dict(zip(md_["start_date"].astype(int), md_["T"].astype(float))))
            except Exception:
                pass
        logmap = _parse_run_logs(folder)        # per-iteration trajectory metrics
        recs = []
        for f in fits:
            try:
                mt = _read_meta(f)
                sd_ = int(float(mt["dataset_start_date"])); md_mode = str(mt["optimize_mode"])
                lm = logmap.get((sd_, md_mode), {})
                recs.append(dict(
                    folder=os.path.basename(folder),
                    expiry=int(float(mt["dataset_expiry_date"])),
                    start_date=sd_,
                    mode=md_mode,
                    S0=float(mt["S0"]),
                    J_init=float(mt["J_initial"]), J_final=float(mt["J_final"]),
                    pct_improve=float(mt["J_pct_improvement"]),
                    true_reduce=float(mt["true_roughness_reduction_pct"]),
                    S_final=float(mt["S_final"]), sqrt_S_final=float(mt["sqrt_S_final"]),
                    run_time=float(mt.get("run_time_sec", np.nan)),
                    n_outer=float(mt.get("n_outer_iterations", np.nan)),
                    T=float(Tmap.get(sd_, np.nan)),
                    iters_to_conv=float(lm.get("iters_to_conv", np.nan)),
                    first_feas=float(lm.get("first_feas", np.nan)),
                    frac_feas=float(lm.get("frac_feas", np.nan)),
                    final_viol=float(lm.get("final_viol", np.nan)),
                    feas_track=lm.get("feas_track", None),
                ))
            except Exception as e:
                print(f"[collect_runs] skip {f}: {e}")
        if prim is None and recs:           # fallback: primary = mode present for EVERY date
            d = pd.DataFrame(recs)
            prim = d.groupby("mode")["start_date"].nunique().idxmax()
        for r in recs:
            r["primary_mode"] = prim
            rows.append(r)
    return pd.DataFrame(rows)


def build_dates(runs, root_key, threshold=THRESHOLD):
    # collapse mode-runs to one winner row per (folder, start_date)
    cfg = ROOT_CONFIG[root_key]
    out = []
    for (folder, sd), g in runs.groupby(["folder", "start_date"]):
        prim_mode = g["primary_mode"].iloc[0]
        pcfg = cfg["pipelines"].get(prim_mode, {"name": prim_mode, "retry": False})
        g = g.sort_values("true_reduce", ascending=False)
        win = g.iloc[0]                                       # winner == best true_reduce
        prim_rows = g[g["mode"] == prim_mode]
        prim = prim_rows.iloc[0] if len(prim_rows) else win   # the primary-mode run
        retry_rows = g[g["mode"] != prim_mode]
        out.append(dict(
            record_type="date", root=root_key, pipeline=pcfg["name"],
            primary_mode=prim_mode, retry_enabled=pcfg["retry"],
            expiry=int(g["expiry"].iloc[0]), start_date=int(sd),
            S0=float(win["S0"]), T=float(win["T"]),
            n_runs=int(len(g)), retried=bool(len(g) > 1),
            retry_mode=(retry_rows["mode"].iloc[0] if len(retry_rows) else ""),
            primary_true=float(prim["true_reduce"]), primary_pct=float(prim["pct_improve"]),
            primary_runtime=float(prim["run_time"]),
            winner_mode=str(win["mode"]), winner_true=float(win["true_reduce"]),
            winner_pct=float(win["pct_improve"]),
            S_final=float(win["S_final"]), sqrt_S_final=float(win["sqrt_S_final"]),
            winner_runtime=float(win["run_time"]), winner_n_outer=float(win["n_outer"]),
            total_runtime=float(g["run_time"].sum()),
            primary_failed=bool(float(prim["true_reduce"]) < threshold),
            winner_failed=bool(float(win["true_reduce"]) < threshold),
        ))
    return (pd.DataFrame(out)
            .sort_values(["pipeline", "expiry", "start_date"]).reset_index(drop=True))


def summarize(dates, threshold=THRESHOLD):
    # one aggregate row per pipeline
    rows = []
    for pname, g in dates.groupby("pipeline", observed=True):
        retry_on = bool(g["retry_enabled"].iloc[0])
        rescue = (g.loc[g["retried"], "winner_true"] - g.loc[g["retried"], "primary_true"])
        rows.append(dict(
            record_type="summary", root=g["root"].iloc[0], pipeline=pname,
            primary_mode=g["primary_mode"].iloc[0], retry_enabled=retry_on,
            n_dates=len(g), n_retries=int(g["retried"].sum()),
            retry_rate=float(g["retried"].mean()),
            n_primary_failed=int(g["primary_failed"].sum()),
            n_winner_failed=int(g["winner_failed"].sum()),
            success_rate=float((~g["winner_failed"]).mean()),
            mean_winner_true=float(g["winner_true"].mean()),
            median_winner_true=float(g["winner_true"].median()),
            std_winner_true=float(g["winner_true"].std()),
            mean_primary_true=float(g["primary_true"].mean()),
            mean_sqrt_S_final=float(g["sqrt_S_final"].mean()),
            median_sqrt_S_final=float(g["sqrt_S_final"].median()),
            std_sqrt_S_final=float(g["sqrt_S_final"].std()),
            mean_winner_runtime=float(g["winner_runtime"].mean()),
            median_winner_runtime=float(g["winner_runtime"].median()),
            sum_winner_runtime=float(g["winner_runtime"].sum()),
            sum_total_runtime=float(g["total_runtime"].sum()),
            mean_total_runtime=float(g["total_runtime"].mean()),
            mean_n_outer=float(g["winner_n_outer"].mean()),
            mean_winner_pct=float(g["winner_pct"].mean()),
            mean_retry_rescue=float(rescue.mean()) if len(rescue) else np.nan,
        ))
    return pd.DataFrame(rows)

In [2]:
def _clip_lo(a, lo=-20):
    # clip extreme negative true-reductions so distribution plots stay readable
    return np.clip(np.asarray(a, float), lo, None)


def _box(ax, data, labels, title):
    ax.boxplot(data, showmeans=True)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
    ax.set_title(title)


def _summary_text(s, retry_on):
    L = [
        f"PIPELINE: {s['pipeline']}",
        f"primary mode: {s['primary_mode']}    retry: {'ON' if retry_on else 'OFF'}",
        "",
        f"# data samples (dates) ........ {int(s['n_dates'])}",
        f"success (true>= {THRESHOLD:.0f}%) ........ "
        f"{int(round(s['success_rate']*s['n_dates']))}/{int(s['n_dates'])} "
        f"({s['success_rate']*100:.1f}%)",
        f"mean true reduction .......... {s['mean_winner_true']:.2f}%",
        f"median true reduction ........ {s['median_winner_true']:.2f}%",
        f"mean roughness sqrt(S_final) .. {s['mean_sqrt_S_final']:.4f}",
        f"mean run-time / date ......... {s['mean_winner_runtime']:.2f} s",
        f"total winner run-time ........ {s['sum_winner_runtime']:.0f} s",
        f"mean outer iterations ........ {s['mean_n_outer']:.0f}",
    ]
    if retry_on:
        L += ["",
              f"# primary FAILED (-> retry) .. {int(s['n_primary_failed'])}",
              f"# retries run ................ {int(s['n_retries'])}",
              f"# retry ALSO failed .......... {int(s['n_winner_failed'])}",
              f"mean retry rescue (true) ..... +{s['mean_retry_rescue']:.2f}%",
              f"total compute incl. retries .. {s['sum_total_runtime']:.0f} s"]
    else:
        L += ["", f"# FAILED (true< {THRESHOLD:.0f}%) ....... {int(s['n_winner_failed'])}"]
    return "\n".join(L)


def _summary_table_text(summ):
    # compact one-row-per-pipeline table; scales to many pipelines
    L = ["pipeline           n   fail  retr |  true% (mean/med/std)  | rough sqrt(S) (mean/med)",
         "-" * 88]
    for _, s in summ.iterrows():
        L.append(f"{_short(s['pipeline']):<16}{int(s['n_dates']):4d}{int(s['n_winner_failed']):5d}"
                 f"{int(s['n_retries']):5d}  |  {s['mean_winner_true']:6.2f} /{s['median_winner_true']:6.2f} /"
                 f"{s['std_winner_true']:6.2f} | {s['mean_sqrt_S_final']:7.4f} /{s['median_sqrt_S_final']:7.4f}")
    L += ["",
          "n=# dates,  fail=# with true<95%,  retr=# retries triggered",
          "run-time (mean s/date):  " + "   ".join(
              f"{_short(s['pipeline'])} {s['mean_winner_runtime']:.1f}" for _, s in summ.iterrows())]
    return "\n".join(L)

In [3]:
def plot_main_single(dates, summ, label, outpath):
    g, s = dates, summ.iloc[0]
    retry_on = bool(g["retry_enabled"].iloc[0])
    fig, ax = plt.subplots(2, 3, figsize=(19, 11))
    fig.suptitle(f"{_sep(label)}core performance",
                 fontsize=15, fontweight="bold")

    ax[0, 0].axis("off")
    ax[0, 0].text(0.0, 1.0, _summary_text(s, retry_on), va="top", ha="left",
                  family="monospace", fontsize=11, transform=ax[0, 0].transAxes)

    if retry_on:
        names = ["# dates", "# retries\n(primary failed)", "# retry\nalso failed"]
        vals  = [s["n_dates"], s["n_retries"], s["n_winner_failed"]]
        cols  = ["#7f7f7f", "#ff7f0e", "#d62728"]
    else:
        names = ["# dates", "# failed\n(<95%)"]
        vals  = [s["n_dates"], s["n_winner_failed"]]
        cols  = ["#7f7f7f", "#d62728"]
    ax[0, 1].bar_label(ax[0, 1].bar(names, vals, color=cols))
    ax[0, 1].set_title("counts")

    ax[0, 2].hist(_clip_lo(g["winner_true"]), bins=30, color=PALETTE[0], alpha=.85)
    ax[0, 2].axvline(THRESHOLD, color="k", ls="--", lw=1.5, label=f"{THRESHOLD:.0f}% threshold")
    ax[0, 2].set_title("true roughness reduction % (winner)")
    ax[0, 2].set_xlabel("true reduction % (clip -20)"); ax[0, 2].legend()

    ax[1, 0].hist(g["sqrt_S_final"], bins=30, color=PALETTE[2], alpha=.85)
    ax[1, 0].set_title("true roughness  sqrt(S_final)")
    ax[1, 0].set_xlabel("sqrt(S_final)  (lower = smoother)")

    ax[1, 1].hist(g["winner_runtime"], bins=30, color=PALETTE[3], alpha=.85)
    ax[1, 1].axvline(g["winner_runtime"].mean(), color="k", ls="--",
                     label=f"mean {g['winner_runtime'].mean():.1f}s")
    ax[1, 1].set_title("run-time per date (winner)")
    ax[1, 1].set_xlabel("seconds"); ax[1, 1].legend()

    ax[1, 2].scatter(g["T"], _clip_lo(g["winner_true"]),
                     c=g["retried"].map({True: "#d62728", False: "#1f77b4"}), s=22)
    ax[1, 2].axhline(THRESHOLD, color="k", ls="--", lw=1)
    ax[1, 2].set_title("true reduction vs tenor T  (red = retried)")
    ax[1, 2].set_xlabel("T (years)"); ax[1, 2].set_ylabel("true reduction %")

    fig.tight_layout(rect=[0, 0, 1, 0.97]); _save(fig, outpath, dpi=110); plt.close(fig)
    return outpath


def plot_extra_single(dates, summ, label, outpath):
    g = dates
    fig, ax = plt.subplots(2, 3, figsize=(19, 11))
    fig.suptitle(f"{_sep(label)}extra diagnostics",
                 fontsize=15, fontweight="bold")

    xs = np.sort(_clip_lo(g["winner_true"]))
    ax[0, 0].plot(xs, np.linspace(0, 1, len(xs)), color=PALETTE[0])
    ax[0, 0].axvline(THRESHOLD, color="k", ls="--", lw=1)
    ax[0, 0].set_title("CDF of true reduction %")
    ax[0, 0].set_xlabel("true reduction %"); ax[0, 0].set_ylabel("fraction <= x")

    ax[0, 1].scatter(g["winner_true"], g["sqrt_S_final"], s=22, color=PALETTE[1], alpha=.7)
    ax[0, 1].set_title("roughness vs true reduction")
    ax[0, 1].set_xlabel("true reduction %"); ax[0, 1].set_ylabel("sqrt(S_final)")

    ax[0, 2].scatter(g["T"], g["winner_runtime"], s=22, color=PALETTE[3], alpha=.7)
    ax[0, 2].set_title("run-time vs tenor T")
    ax[0, 2].set_xlabel("T (years)"); ax[0, 2].set_ylabel("seconds")

    ax[1, 0].hist(g["winner_n_outer"].dropna(), bins=25, color="#8c564b", alpha=.85)
    ax[1, 0].set_title("# outer iterations (winner)"); ax[1, 0].set_xlabel("outer iterations")

    per_exp = g.groupby("expiry")["winner_true"].mean()
    ax[1, 1].plot(range(len(per_exp)), per_exp.values, "o-", ms=4, color=PALETTE[2])
    ax[1, 1].axhline(THRESHOLD, color="k", ls="--", lw=1)
    ax[1, 1].set_title("mean true reduction by expiry")
    ax[1, 1].set_xlabel("expiry index (chron.)"); ax[1, 1].set_ylabel("mean true reduction %")

    rr = g[g["retried"]].reset_index(drop=True)
    if len(rr):
        idx = np.arange(len(rr))
        ax[1, 2].scatter(idx, _clip_lo(rr["primary_true"]), color="#ff7f0e", label="primary", s=30)
        ax[1, 2].scatter(idx, _clip_lo(rr["winner_true"]),  color="#2ca02c", label="kept (winner)", s=30)
        for i in idx:
            ax[1, 2].plot([i, i], [_clip_lo([rr["primary_true"][i]])[0],
                                   _clip_lo([rr["winner_true"][i]])[0]], color="gray", lw=.8, zorder=0)
        ax[1, 2].axhline(THRESHOLD, color="k", ls="--", lw=1)
        ax[1, 2].set_title(f"retry rescue ({len(rr)} retried dates)")
        ax[1, 2].set_xlabel("retried-date index"); ax[1, 2].set_ylabel("true reduction % (clip -20)")
        ax[1, 2].legend()
    else:
        ax[1, 2].hist(g["winner_pct"], bins=25, color="#17becf", alpha=.85)
        ax[1, 2].set_title("log-J improvement % (no retries occurred)")
        ax[1, 2].set_xlabel("log-J improvement %")

    fig.tight_layout(rect=[0, 0, 1, 0.97]); _save(fig, outpath, dpi=110); plt.close(fig)
    return outpath

In [4]:
def plot_main_compare(dates, summ, label, outpath):
    pipes = list(summ["pipeline"])
    fig, ax = plt.subplots(2, 3, figsize=(19, 11))
    fig.suptitle(f"{_sep(label)}core performance comparison",
                 fontsize=15, fontweight="bold")

    ax[0, 0].axis("off")
    ax[0, 0].text(0.0, 1.0, _summary_table_text(summ), va="top", ha="left",
                  family="monospace", fontsize=9, transform=ax[0, 0].transAxes)

    metrics = ["n_dates", "n_primary_failed", "n_retries", "n_winner_failed"]
    mlabels = ["# dates", "# primary\nfailed", "# retries", "# winner\nfailed"]
    x = np.arange(len(metrics)); w = 0.8 / len(pipes)
    for j, (_, s) in enumerate(summ.iterrows()):
        ax[0, 1].bar(x + j * w, [s[m] for m in metrics], width=w,
                     label=_short(s["pipeline"]), color=PALETTE[j])
    ax[0, 1].set_xticks(x + w * (len(pipes) - 1) / 2); ax[0, 1].set_xticklabels(mlabels)
    ax[0, 1].set_title("counts"); ax[0, 1].legend(fontsize=7)

    lab = [_short(p) for p in pipes]
    _box(ax[0, 2], [_clip_lo(dates[dates.pipeline == p]["winner_true"]) for p in pipes],
         lab, "true reduction % (clip -20)"); ax[0, 2].axhline(THRESHOLD, color="k", ls="--", lw=1)
    _box(ax[1, 0], [dates[dates.pipeline == p]["sqrt_S_final"] for p in pipes],
         lab, "roughness sqrt(S_final)")
    _box(ax[1, 1], [dates[dates.pipeline == p]["winner_runtime"] for p in pipes],
         lab, "run-time per date (s)")

    for j, p in enumerate(pipes):
        gg = dates[dates.pipeline == p]
        ax[1, 2].scatter(gg["T"], _clip_lo(gg["winner_true"]), s=20, alpha=.6,
                         color=PALETTE[j], rasterized=True, label=lab[j])
    ax[1, 2].axhline(THRESHOLD, color="k", ls="--", lw=1)
    ax[1, 2].set_title("true reduction vs tenor T")
    ax[1, 2].set_xlabel("T (years)"); ax[1, 2].legend(fontsize=8)

    fig.tight_layout(rect=[0, 0, 1, 0.97]); _save(fig, outpath, dpi=110); plt.close(fig)
    return outpath


def plot_extra_compare(dates, summ, label, outpath):
    pipes = list(summ["pipeline"]); lab = [_short(p) for p in pipes]
    fig, ax = plt.subplots(2, 3, figsize=(19, 11))
    fig.suptitle(f"{_sep(label)}extra diagnostics & paired comparison",
                 fontsize=15, fontweight="bold")

    for j, p in enumerate(pipes):
        xs = np.sort(_clip_lo(dates[dates.pipeline == p]["winner_true"]))
        ax[0, 0].plot(xs, np.linspace(0, 1, len(xs)), color=PALETTE[j], label=lab[j])
    ax[0, 0].axvline(THRESHOLD, color="k", ls="--", lw=1)
    ax[0, 0].set_title("CDF of true reduction %"); ax[0, 0].set_xlabel("true reduction %")
    ax[0, 0].legend(fontsize=8)

    # paired deltas vs a baseline pipeline (a standalone sig_only if present, else no-retry, else first)
    base = next((p for p in pipes if p in ("sig_only [1]", "sig_only")),
                next((p for p in pipes if "no retry" in p), pipes[0]))
    others = [p for p in pipes if p != base]
    blab = _short(base)
    db = dates[dates.pipeline == base].set_index(["expiry", "start_date"])
    for k, p in enumerate(others):
        dp = dates[dates.pipeline == p].set_index(["expiry", "start_date"])
        shared = dp.index.intersection(db.index)
        col = PALETTE[pipes.index(p)]
        d_true = (dp.loc[shared, "winner_true"] - db.loc[shared, "winner_true"]).values
        ax[0, 1].hist(np.clip(d_true, -20, 20), bins=30, alpha=.5, color=col,
                      label=f"{_short(p)} − {blab}")
        d_rt = (dp.loc[shared, "winner_runtime"] - db.loc[shared, "winner_runtime"]).values
        ax[0, 2].hist(d_rt, bins=30, alpha=.5, color=col, label=f"{_short(p)} − {blab}")
    ax[0, 1].axvline(0, color="k", lw=1)
    ax[0, 1].set_title(f"paired Δtrue vs {blab}  (>0 favours other)")
    ax[0, 1].set_xlabel("Δ true reduction % (clip ±20)"); ax[0, 1].legend(fontsize=8)
    ax[0, 2].axvline(0, color="k", lw=1)
    ax[0, 2].set_title(f"paired Δrun-time vs {blab}")
    ax[0, 2].set_xlabel("Δ seconds"); ax[0, 2].legend(fontsize=8)

    for j, p in enumerate(pipes):
        gg = dates[dates.pipeline == p]
        ax[1, 0].scatter(gg["winner_true"], gg["sqrt_S_final"], s=18, alpha=.55,
                         color=PALETTE[j], rasterized=True, label=lab[j])
    ax[1, 0].set_title("roughness vs true reduction")
    ax[1, 0].set_xlabel("true reduction %"); ax[1, 0].set_ylabel("sqrt(S_final)")
    ax[1, 0].legend(fontsize=8)

    for j, p in enumerate(pipes):
        gg = dates[dates.pipeline == p]
        ax[1, 1].scatter(gg["T"], gg["winner_runtime"], s=18, alpha=.55,
                         color=PALETTE[j], rasterized=True, label=lab[j])
    ax[1, 1].set_title("run-time vs tenor T")
    ax[1, 1].set_xlabel("T (years)"); ax[1, 1].set_ylabel("seconds"); ax[1, 1].legend(fontsize=8)

    for j, p in enumerate(pipes):
        per_exp = dates[dates.pipeline == p].groupby("expiry")["winner_true"].mean()
        ax[1, 2].plot(range(len(per_exp)), per_exp.values, "o-", ms=3,
                      color=PALETTE[j], label=lab[j])
    ax[1, 2].axhline(THRESHOLD, color="k", ls="--", lw=1)
    ax[1, 2].set_title("mean true reduction by expiry")
    ax[1, 2].set_xlabel("expiry index (chron.)"); ax[1, 2].legend(fontsize=8)

    fig.tight_layout(rect=[0, 0, 1, 0.97]); _save(fig, outpath, dpi=110); plt.close(fig)
    return outpath

In [5]:
def _robust_ylim(ax, series_list, pad=1.15):
    # set an upper y-limit from the 97.5th pctile so a few outliers do not squash the plot
    vals = np.concatenate([np.asarray(s, float) for s in series_list]) if series_list else np.array([0.0])
    hi = np.nanpercentile(vals, 97.5) * pad
    lo = min(0.0, np.nanmin(vals))
    if np.isfinite(hi) and hi > lo:
        ax.set_ylim(lo, hi)


def pipeline_stats_table(dates):
    """Per-pipeline mean/median/std of true reduction %, roughness sqrt(S_final) and run-time.
    Single source of truth for both the in-figure stats panel and the LaTeX table."""
    cats = (list(dates["pipeline"].cat.categories)
            if hasattr(dates["pipeline"], "cat") else list(dict.fromkeys(dates["pipeline"])))
    rows = []
    for p in cats:
        g = dates[dates.pipeline == p]
        if len(g) == 0:
            continue
        t, r, rt, ic = g["winner_true"], g["sqrt_S_final"], g["winner_runtime"], g["iters_to_conv"]
        rows.append(dict(
            pipeline=_short(p), n=int(len(g)), fail=int((t < THRESHOLD).sum()),
            fail_pct=float(100.0 * (t < THRESHOLD).sum() / len(g)),
            true_mean=float(t.mean()), true_med=float(t.median()), true_std=float(t.std()),
            rough_mean=float(r.mean()), rough_med=float(r.median()), rough_std=float(r.std()),
            rt_mean=float(rt.mean()), rt_med=float(rt.median()), rt_std=float(rt.std()),
            conv_mean=float(ic.mean()), conv_med=float(ic.median()), conv_std=float(ic.std()),
        ))
    return pd.DataFrame(rows)


def _stats_text(st):
    # aligned monospace block: name col width 20, three number cols width 9 each
    W = 20
    def block(title, cols, nd):
        L = [title, f"{'':<{W}}{'mean':>9}{'median':>9}{'std':>9}"]
        for _, r in st.iterrows():
            L.append(f"{r['pipeline']:<{W}}" + "".join(f"{r[c]:>9.{nd}f}" for c in cols))
        return L
    L  = [f"n = {int(st['n'].iloc[0])} samples per pipeline", ""]
    L += block("TRUE REDUCTION %", ["true_mean", "true_med", "true_std"], 2) + [""]
    L += block("ROUGHNESS  sqrt(S_final)", ["rough_mean", "rough_med", "rough_std"], 4) + [""]
    L += block("RUN-TIME  (s per date)", ["rt_mean", "rt_med", "rt_std"], 2) + [""]
    L += block("ITERS TO CONVERGE", ["conv_mean", "conv_med", "conv_std"], 1) + [""]
    L += [f"FAILED  (true < {THRESHOLD:.0f}%)", f"{'':<{W}}{'#':>9}{'%':>9}"]
    for _, r in st.iterrows():
        L.append(f"{r['pipeline']:<{W}}{int(r['fail']):>9d}{r['fail_pct']:>9.1f}")
    return "\n".join(L)


def stats_latex(st, caption="2011--2012 optimization pipelines: per-pipeline statistics "
                            "(true roughness reduction \\%, roughness $\\sqrt{S_{\\mathrm{final}}}$, "
                            "run-time in seconds). Requires \\usepackage{booktabs}.",
                label="tab:pipeline_stats"):
    fmts = [("n", "{:d}"), ("fail", "{:d}"), ("fail_pct", "{:.1f}"),
            ("true_mean", "{:.2f}"), ("true_med", "{:.2f}"), ("true_std", "{:.2f}"),
            ("rough_mean", "{:.4f}"), ("rough_med", "{:.4f}"), ("rough_std", "{:.4f}"),
            ("rt_mean", "{:.2f}"), ("rt_med", "{:.2f}"), ("rt_std", "{:.2f}"),
            ("conv_mean", "{:.1f}"), ("conv_med", "{:.1f}"), ("conv_std", "{:.1f}")]
    body = []
    for _, r in st.iterrows():
        cells = [str(r["pipeline"]).replace("_", r"\_")] + [f.format(r[k]) for k, f in fmts]
        body.append("  " + " & ".join(cells) + r" \\")
    out = [r"\begin{table}[ht]\centering",
           r"\caption{" + caption + "}", r"\label{" + label + "}",
           r"\begin{tabular}{l" + "r" * len(fmts) + "}", r"\toprule",
           r" & & & & \multicolumn{3}{c}{true reduction \%}"
           r" & \multicolumn{3}{c}{roughness $\sqrt{S_{\mathrm{final}}}$}"
           r" & \multicolumn{3}{c}{run-time (s)}"
           r" & \multicolumn{3}{c}{iters to converge} \\",
           r"\cmidrule(lr){5-7}\cmidrule(lr){8-10}\cmidrule(lr){11-13}\cmidrule(lr){14-16}",
           r"Pipeline & $n$ & \#fail & \#fail\% & mean & median & std & mean & median & std"
           r" & mean & median & std & mean & median & std \\", r"\midrule"]
    out += body
    out += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(out)


def plot_compare_grid(dates, summ, label, outpath):
    """row 0: roughness / run-time / true-reduction vs data-point index;
    row 1: same vs time-to-expiry;
    row 2: initial roughness S_init vs index & vs T (shared by all pipelines), mean true
           reduction by expiry;
    row 3: two 95-100% zoom-ins of true reduction + a mean/median/std stats table;
    row 4: CDF of true reduction (zoom 95-100%, full range) + roughness-vs-true (zoom 95-100%);
    row 5: failure counts for all pipelines."""
    pipes = list(summ["pipeline"]); lab = [_short(p) for p in pipes]
    mk = _markers(pipes); zo = _zorders(pipes)   # distinct markers for single-mode pipelines, "o" for retry ones

    # canonical data-point ordering shared by all pipelines: sort by (expiry, start_date)
    keys = sorted(set(zip(dates["expiry"], dates["start_date"])))
    kidx = {k: i for i, k in enumerate(keys)}

    def with_index(p):
        g = dates[dates.pipeline == p].copy()
        g["xi"] = [kidx[(e, s)] for e, s in zip(g["expiry"], g["start_date"])]
        return g.sort_values("xi")

    fig = plt.figure(figsize=(20, 29))
    gs = fig.add_gridspec(6, 3, height_ratios=[1, 1, 1, 1, 1, 0.85])
    fig.suptitle(f"{_sep(label)}{len(pipes)}-way comparison", fontsize=16, fontweight="bold")

    metrics = [("sqrt_S_final", "true roughness  sqrt(S_final)", False),
               ("winner_runtime", "run-time per date (s)", False),
               ("winner_true", "true roughness reduction %", True)]

    # ── row 0: metric vs data-point index ──
    for c, (col, title, is_true) in enumerate(metrics):
        ax = fig.add_subplot(gs[0, c]); ser = []
        for j, p in enumerate(pipes):
            g = with_index(p)
            y = _clip_lo(g[col]) if is_true else g[col].values
            ax.scatter(g["xi"], y, s=8, alpha=.55, color=PALETTE[j], marker=mk[j], zorder=zo[j], rasterized=True, label=lab[j]); ser.append(y)
        if is_true:
            ax.axhline(THRESHOLD, color="k", ls="--", lw=1)
            ax.set_ylim(min(-20, np.nanmin(np.concatenate([np.asarray(r) for r in ser]))), 102)
        elif col == "sqrt_S_final":
            _robust_ylim(ax, ser)
        ax.set_title(f"{title}  vs  data-point index")
        ax.set_xlabel("data sample index (sorted by expiry, start date)")
        if c == 0: ax.legend(fontsize=8)

    # ── row 1: metric vs time-to-expiry T ──
    for c, (col, title, is_true) in enumerate(metrics):
        ax = fig.add_subplot(gs[1, c]); rs = []
        for j, p in enumerate(pipes):
            g = dates[dates.pipeline == p]
            y = _clip_lo(g[col]) if is_true else g[col].values
            ax.scatter(g["T"], y, s=12, alpha=.55, color=PALETTE[j], marker=mk[j], zorder=zo[j], rasterized=True, label=lab[j]); rs.append(y)
        if is_true:
            ax.axhline(THRESHOLD, color="k", ls="--", lw=1)
            ax.set_ylim(min(-20, np.nanmin(np.concatenate([np.asarray(r) for r in rs]))), 102)
        elif col == "sqrt_S_final":
            _robust_ylim(ax, rs)
        ax.set_title(f"{title}  vs  time-to-expiry")
        ax.set_xlabel("time to expiry  T (years)")
        if c == 0: ax.legend(fontsize=8)

    # ── row 2: initial roughness sqrt(S_init) (same for all pipelines) + mean true by expiry ──
    base1 = with_index(pipes[0]).drop_duplicates("xi")   # one row per data sample
    axsi = fig.add_subplot(gs[2, 0])
    axsi.scatter(base1["xi"], base1["sqrt_S_init"], s=10, alpha=.7, color="#444444")
    _robust_ylim(axsi, [base1["sqrt_S_init"].values])
    axsi.set_title("initial roughness  sqrt(S_init) = exp(J_init/2)  vs  data-point index")
    axsi.set_xlabel("data sample index (sorted by expiry, start date)")
    axsi.set_ylabel("sqrt(S_init)")

    axst = fig.add_subplot(gs[2, 1])
    axst.scatter(base1["T"], base1["sqrt_S_init"], s=12, alpha=.7, color="#444444")
    _robust_ylim(axst, [base1["sqrt_S_init"].values])
    axst.set_title("initial roughness  sqrt(S_init) = exp(J_init/2)  vs  time-to-expiry")
    axst.set_xlabel("time to expiry  T (years)"); axst.set_ylabel("sqrt(S_init)")

    axme = fig.add_subplot(gs[2, 2])
    for j, p in enumerate(pipes):
        per_exp = dates[dates.pipeline == p].groupby("expiry")["winner_true"].mean()
        axme.scatter(range(len(per_exp)), per_exp.values, s=16, color=PALETTE[j], marker=mk[j], zorder=zo[j], rasterized=True, label=lab[j])
    axme.axhline(THRESHOLD, color="k", ls="--", lw=1)
    axme.set_title("mean true reduction %  by expiry")
    axme.set_xlabel("expiry index (chronological)"); axme.set_ylabel("mean true reduction %")
    axme.legend(fontsize=7)

    # ── row 3: zoom-ins on the 95-100% true-reduction band ──
    azi = fig.add_subplot(gs[3, 0])
    for j, p in enumerate(pipes):
        g = with_index(p)
        azi.scatter(g["xi"], g["winner_true"], s=10, alpha=.6, color=PALETTE[j], marker=mk[j], zorder=zo[j], rasterized=True, label=lab[j])
    azi.set_ylim(95, 100); azi.axhline(THRESHOLD, color="k", ls="--", lw=1)
    azi.set_title("true reduction %  vs  data-point index  (ZOOM 95-100%)")
    azi.set_xlabel("data sample index (sorted by expiry, start date)")
    azi.set_ylabel("true reduction %"); azi.legend(fontsize=8)

    azt = fig.add_subplot(gs[3, 1])
    for j, p in enumerate(pipes):
        g = dates[dates.pipeline == p]
        azt.scatter(g["T"], g["winner_true"], s=12, alpha=.6, color=PALETTE[j], marker=mk[j], zorder=zo[j], rasterized=True, label=lab[j])
    azt.set_ylim(95, 100); azt.axhline(THRESHOLD, color="k", ls="--", lw=1)
    azt.set_title("true reduction %  vs  time-to-expiry  (ZOOM 95-100%)")
    azt.set_xlabel("time to expiry  T (years)")
    azt.set_ylabel("true reduction %"); azt.legend(fontsize=8)

    # (3,2): roughness vs true reduction (FULL range) — the stats table now lives in its own image
    arf = fig.add_subplot(gs[3, 2])
    for j, p in enumerate(pipes):
        g = dates[dates.pipeline == p]
        arf.scatter(g["winner_true"], g["sqrt_S_final"], s=14, alpha=.55, color=PALETTE[j], marker=mk[j], zorder=zo[j], rasterized=True, label=lab[j])
    _robust_ylim(arf, [dates["sqrt_S_final"].values])
    arf.set_title("roughness sqrt(S_final)  vs  true reduction %  (full range)")
    arf.set_xlabel("true reduction %"); arf.set_ylabel("sqrt(S_final)"); arf.legend(fontsize=7)

    # ── row 4: CDF of true reduction (zoom 99-100% & full) + roughness-vs-true zoom ──
    acz = fig.add_subplot(gs[4, 0])
    acf = fig.add_subplot(gs[4, 1])
    for j, p in enumerate(pipes):
        xs = np.sort(dates[dates.pipeline == p]["winner_true"].values)
        ys = np.linspace(0, 1, len(xs))
        acz.plot(xs, ys, color=PALETTE[j], label=lab[j])
        acf.plot(_clip_lo(xs), ys, color=PALETTE[j], label=lab[j])
    acz.set_xlim(99, 100); acz.set_ylim(0.0, 1.002)
    acz.axvline(THRESHOLD, color="k", ls="--", lw=1)
    acz.set_title("CDF of true reduction %  (ZOOM 99-100%)")
    acz.set_xlabel("true reduction %"); acz.set_ylabel("fraction <= x"); acz.legend(fontsize=8)
    acf.axvline(THRESHOLD, color="k", ls="--", lw=1)
    acf.set_title("CDF of true reduction %  (full range, clip -20)")
    acf.set_xlabel("true reduction %"); acf.set_ylabel("fraction <= x"); acf.legend(fontsize=8)

    arz = fig.add_subplot(gs[4, 2])
    for j, p in enumerate(pipes):
        g = dates[dates.pipeline == p]
        arz.scatter(g["winner_true"], g["sqrt_S_final"], s=14, alpha=.55,
                    color=PALETTE[j], marker=mk[j], zorder=zo[j], rasterized=True, label=lab[j])
    arz.set_xlim(95, 100)
    sub = dates[dates["winner_true"] >= 95]["sqrt_S_final"]
    arz.set_ylim(0, float(np.nanpercentile(sub, 99)) * 1.1 if len(sub) else 1.0)
    arz.set_title("roughness sqrt(S_final)  vs  true reduction %  (ZOOM 95-100%)")
    arz.set_xlabel("true reduction %"); arz.set_ylabel("sqrt(S_final)"); arz.legend(fontsize=8)

    # ── row 5 (spans all columns): final failure count per pipeline (true < 95%) ──
    axf = fig.add_subplot(gs[5, :])
    x = np.arange(len(pipes))
    finl = [int(summ.iloc[i]["n_winner_failed"]) for i in range(len(pipes))]
    ndts = [int(summ.iloc[i]["n_dates"]) for i in range(len(pipes))]
    fpct = [100.0 * f / n for f, n in zip(finl, ndts)]
    bars = axf.bar(x, finl, 0.6, color=[PALETTE[i] for i in range(len(pipes))])
    axf.bar_label(bars, labels=[f"{f}\n{p:.1f}%" for f, p in zip(finl, fpct)])
    axf.set_xticks(x)
    axf.set_xticklabels([f"{_short(p)}\n(n={n})" for p, n in zip(pipes, ndts)])
    axf.set_ylabel("# failed dates  (true < 95%)")
    axf.set_title(f"failure counts & failed %  (threshold = true reduction < {THRESHOLD:.0f}%)  "
                  f"— compare each primary-only vs its +retry neighbour")

    fig.tight_layout(rect=[0, 0, 1, 0.97]); _save(fig, outpath, dpi=110); plt.close(fig)
    return outpath


# ──────────────── per-plot draw helpers (used by plot_compare_panels) ────────────────
def _ck_index(dates):
    keys = sorted(set(zip(dates["expiry"], dates["start_date"])))
    return {k: i for i, k in enumerate(keys)}

def _pi(dates, p, kidx):
    g = dates[dates.pipeline == p].copy()
    g["xi"] = [kidx[(e, s)] for e, s in zip(g["expiry"], g["start_date"])]
    return g.sort_values("xi")

def _xlabel(xkind):
    return ("data sample index (sorted by expiry, start date)" if xkind == "index"
            else "time to expiry  T (years)")

def _d_metric(ax, dates, pipes, lab, col, title, is_true, xkind):
    kidx = _ck_index(dates) if xkind == "index" else None
    mk = _markers(pipes); zo = _zorders(pipes); ser = []
    for j, p in enumerate(pipes):
        g = _pi(dates, p, kidx) if xkind == "index" else dates[dates.pipeline == p]
        x = g["xi"] if xkind == "index" else g["T"]
        y = _clip_lo(g[col]) if is_true else g[col].values
        ax.scatter(x, y, s=10, alpha=.55, color=PALETTE[j], marker=mk[j], zorder=zo[j], rasterized=True, label=lab[j]); ser.append(y)
    if is_true:
        ax.axhline(THRESHOLD, color="k", ls="--", lw=1)
        ax.set_ylim(min(-20, np.nanmin(np.concatenate([np.asarray(r) for r in ser]))), 102)
    elif col == "sqrt_S_final":
        _robust_ylim(ax, ser)
    nm = "data-point index" if xkind == "index" else "time-to-expiry"
    ax.set_title(f"{title}  vs  {nm}"); ax.set_xlabel(_xlabel(xkind)); ax.legend(fontsize=8)

def _d_true_zoom(ax, dates, pipes, lab, xkind):
    kidx = _ck_index(dates) if xkind == "index" else None
    mk = _markers(pipes); zo = _zorders(pipes)
    for j, p in enumerate(pipes):
        g = _pi(dates, p, kidx) if xkind == "index" else dates[dates.pipeline == p]
        x = g["xi"] if xkind == "index" else g["T"]
        ax.scatter(x, g["winner_true"], s=10, alpha=.6, color=PALETTE[j], marker=mk[j], zorder=zo[j], rasterized=True, label=lab[j])
    ax.set_ylim(95, 100); ax.axhline(THRESHOLD, color="k", ls="--", lw=1)
    nm = "data-point index" if xkind == "index" else "time-to-expiry"
    ax.set_title(f"true reduction %  vs  {nm}  (ZOOM 95-100%)")
    ax.set_xlabel(_xlabel(xkind)); ax.set_ylabel("true reduction %"); ax.legend(fontsize=8)

def _d_sinit(ax, dates, xkind):
    # TRUE initial roughness = sqrt(S_init) = exp(J_init/2)
    kidx = _ck_index(dates)
    base1 = _pi(dates, dates["pipeline"].iloc[0], kidx).drop_duplicates("xi")
    x = base1["xi"] if xkind == "index" else base1["T"]
    ax.scatter(x, base1["sqrt_S_init"], s=11, alpha=.7, color="#444444")
    _robust_ylim(ax, [base1["sqrt_S_init"].values])
    nm = "data-point index" if xkind == "index" else "time-to-expiry"
    ax.set_title(f"initial roughness  sqrt(S_init) = exp(J_init/2)  vs  {nm}  (same for all pipelines)")
    ax.set_xlabel(_xlabel(xkind)); ax.set_ylabel("sqrt(S_init)")

def _d_sfinal(ax, dates, pipes, lab, xkind):
    # TRUE final roughness = sqrt(S_final) = exp(J_final/2), coloured per pipeline
    kidx = _ck_index(dates) if xkind == "index" else None
    mk = _markers(pipes); zo = _zorders(pipes); ser = []
    for j, p in enumerate(pipes):
        g = _pi(dates, p, kidx) if xkind == "index" else dates[dates.pipeline == p]
        x = g["xi"] if xkind == "index" else g["T"]
        ax.scatter(x, g["sqrt_S_final"], s=10, alpha=.55, color=PALETTE[j], marker=mk[j], zorder=zo[j], rasterized=True, label=lab[j])
        ser.append(g["sqrt_S_final"].values)
    _robust_ylim(ax, ser)
    nm = "data-point index" if xkind == "index" else "time-to-expiry"
    ax.set_title(f"final roughness  sqrt(S_final) = exp(J_final/2)  vs  {nm}")
    ax.set_xlabel(_xlabel(xkind)); ax.set_ylabel("sqrt(S_final)"); ax.legend(fontsize=8)

def _d_sinit_vs_sfinal(ax, dates, pipes, lab, xkind):
    # overlay the shared INITIAL roughness sqrt(S_init) (grey 'x') with each pipeline's FINAL
    # roughness sqrt(S_final) (coloured dots) -> shows the before/after reduction on one axis
    kidx = _ck_index(dates) if xkind == "index" else None
    base = _pi(dates, pipes[0], kidx) if xkind == "index" else dates[dates.pipeline == pipes[0]]
    if xkind == "index":
        base = base.drop_duplicates("xi")
    xb = base["xi"] if xkind == "index" else base["T"]
    mk = _markers(pipes); zo = _zorders(pipes); ser = [base["sqrt_S_init"].values]
    ax.scatter(xb, base["sqrt_S_init"], s=30, marker="*", linewidths=0.6, rasterized=True,
               color="#333333", alpha=.65, label="sqrt(S_init)  initial (shared)", zorder=3)
    for j, p in enumerate(pipes):
        g = _pi(dates, p, kidx) if xkind == "index" else dates[dates.pipeline == p]
        x = g["xi"] if xkind == "index" else g["T"]
        ax.scatter(x, g["sqrt_S_final"], s=9, alpha=.6, color=PALETTE[j], marker=mk[j],
                   rasterized=True, label=f"{lab[j]}  final", zorder=2)
        ser.append(g["sqrt_S_final"].values)
    _robust_ylim(ax, ser)
    nm = "data-point index" if xkind == "index" else "time-to-expiry"
    ax.set_title(f"roughness sqrt(S): initial (grey *) vs final (coloured)  vs  {nm}")
    ax.set_xlabel(_xlabel(xkind)); ax.set_ylabel("sqrt(S)"); ax.legend(fontsize=6, ncol=2)

def _d_mean_by_expiry(ax, dates, pipes, lab):
    # scatter only (no connecting lines)
    mk = _markers(pipes); zo = _zorders(pipes)
    for j, p in enumerate(pipes):
        per_exp = dates[dates.pipeline == p].groupby("expiry")["winner_true"].mean()
        ax.scatter(range(len(per_exp)), per_exp.values, s=16, color=PALETTE[j], marker=mk[j], zorder=zo[j], rasterized=True, label=lab[j])
    ax.axhline(THRESHOLD, color="k", ls="--", lw=1)
    ax.set_title("mean true reduction %  by expiry")
    ax.set_xlabel("expiry index (chronological)"); ax.set_ylabel("mean true reduction %")
    ax.legend(fontsize=8)

def _d_cdf(ax, dates, pipes, lab, zoom):
    for j, p in enumerate(pipes):
        xs = np.sort(dates[dates.pipeline == p]["winner_true"].values)
        ys = np.linspace(0, 1, len(xs))
        ax.plot(xs if zoom else _clip_lo(xs), ys, color=PALETTE[j], label=lab[j])
    ax.axvline(THRESHOLD, color="k", ls="--", lw=1)
    if zoom:
        ax.set_xlim(99, 100); ax.set_ylim(0.0, 1.002)
        ax.set_title("CDF of true reduction %  (ZOOM 99-100%)")
    else:
        ax.set_title("CDF of true reduction %  (full range, clip -20)")
    ax.set_xlabel("true reduction %"); ax.set_ylabel("fraction <= x"); ax.legend(fontsize=8)

def _d_rough_true(ax, dates, pipes, lab, zoom):
    mk = _markers(pipes); zo = _zorders(pipes)
    for j, p in enumerate(pipes):
        g = dates[dates.pipeline == p]
        ax.scatter(g["winner_true"], g["sqrt_S_final"], s=14, alpha=.55,
                   color=PALETTE[j], marker=mk[j], zorder=zo[j], rasterized=True, label=lab[j])
    if zoom:
        ax.set_xlim(95, 100)
        sub = dates[dates["winner_true"] >= 95]["sqrt_S_final"]
        ax.set_ylim(0, float(np.nanpercentile(sub, 99)) * 1.1 if len(sub) else 1.0)
        ax.set_title("roughness sqrt(S_final)  vs  true reduction %  (ZOOM 95-100%)")
    else:
        _robust_ylim(ax, [dates["sqrt_S_final"].values])
        ax.set_title("roughness sqrt(S_final)  vs  true reduction %  (full range)")
    ax.set_xlabel("true reduction %"); ax.set_ylabel("sqrt(S_final)"); ax.legend(fontsize=8)

def _d_failures(ax, summ, pipes, lab):
    x = np.arange(len(pipes))
    finl = [int(summ.iloc[i]["n_winner_failed"]) for i in range(len(pipes))]
    ndts = [int(summ.iloc[i]["n_dates"]) for i in range(len(pipes))]
    fpct = [100.0 * f / n for f, n in zip(finl, ndts)]
    bars = ax.bar(x, finl, 0.6, color=[PALETTE[i] for i in range(len(pipes))])
    ax.bar_label(bars, labels=[f"{f}\n{p:.1f}%" for f, p in zip(finl, fpct)])
    ax.set_xticks(x); ax.set_xticklabels([f"{l}\n(n={n})" for l, n in zip(lab, ndts)])
    ax.set_ylabel("# failed dates  (true < 95%)")
    ax.set_title(f"failure counts & failed %  (threshold = true reduction < {THRESHOLD:.0f}%)")


def _d_fail_pct(ax, summ, pipes, lab):
    x = np.arange(len(pipes))
    finl = [int(summ.iloc[i]["n_winner_failed"]) for i in range(len(pipes))]
    ndts = [int(summ.iloc[i]["n_dates"]) for i in range(len(pipes))]
    fpct = [100.0 * f / n for f, n in zip(finl, ndts)]
    bars = ax.bar(x, fpct, 0.6, color=[PALETTE[i] for i in range(len(pipes))])
    ax.bar_label(bars, labels=[f"{p:.1f}%" for p in fpct])
    ax.set_xticks(x); ax.set_xticklabels([f"{l}\n(n={n})" for l, n in zip(lab, ndts)])
    ax.set_ylabel("failed %  (true < 95%)")
    ax.set_title(f"failed percentage  (threshold = true reduction < {THRESHOLD:.0f}%)")


def plot_compare_panels(dates, summ, label, out_dir):
    """Save every plot of the compare figure as its OWN image; the non-zoomed and zoomed
    versions of a plot are paired side-by-side in a single image."""
    os.makedirs(out_dir, exist_ok=True)
    pipes = list(summ["pipeline"]); lab = [_short(p) for p in pipes]
    saved = []

    def save(name, draws):
        n = len(draws)
        fig, axs = plt.subplots(1, n, figsize=(8.6 * n, 6.0), squeeze=False)
        for ax, d in zip(axs[0], draws):
            d(ax)
        fig.suptitle(label, fontsize=12, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.94])
        fp = os.path.join(out_dir, f"{name}.{IMG_EXT}")
        _save(fig, fp); plt.close(fig); saved.append(fp)

    save("roughness_vs_index",
         [lambda ax: _d_metric(ax, dates, pipes, lab, "sqrt_S_final", "roughness sqrt(S_final)", False, "index")])
    save("roughness_vs_tenor",
         [lambda ax: _d_metric(ax, dates, pipes, lab, "sqrt_S_final", "roughness sqrt(S_final)", False, "T")])
    save("runtime_vs_index",
         [lambda ax: _d_metric(ax, dates, pipes, lab, "winner_runtime", "run-time per date (s)", False, "index")])
    save("runtime_vs_tenor",
         [lambda ax: _d_metric(ax, dates, pipes, lab, "winner_runtime", "run-time per date (s)", False, "T")])
    # non-zoomed + zoomed paired in one image:
    save("true_reduction_vs_index",
         [lambda ax: _d_metric(ax, dates, pipes, lab, "winner_true", "true reduction %", True, "index"),
          lambda ax: _d_true_zoom(ax, dates, pipes, lab, "index")])
    save("true_reduction_vs_tenor",
         [lambda ax: _d_metric(ax, dates, pipes, lab, "winner_true", "true reduction %", True, "T"),
          lambda ax: _d_true_zoom(ax, dates, pipes, lab, "T")])
    save("S_init_vs_index", [lambda ax: _d_sinit(ax, dates, "index")])
    save("S_init_vs_tenor", [lambda ax: _d_sinit(ax, dates, "T")])
    save("S_final_vs_index", [lambda ax: _d_sfinal(ax, dates, pipes, lab, "index")])
    save("S_final_vs_tenor", [lambda ax: _d_sfinal(ax, dates, pipes, lab, "T")])
    save("sqrt_S_init_vs_final_index", [lambda ax: _d_sinit_vs_sfinal(ax, dates, pipes, lab, "index")])
    save("sqrt_S_init_vs_final_tenor", [lambda ax: _d_sinit_vs_sfinal(ax, dates, pipes, lab, "T")])
    save("mean_true_by_expiry", [lambda ax: _d_mean_by_expiry(ax, dates, pipes, lab)])
    save("cdf_true_reduction",
         [lambda ax: _d_cdf(ax, dates, pipes, lab, False),
          lambda ax: _d_cdf(ax, dates, pipes, lab, True)])
    save("roughness_vs_true_reduction",
         [lambda ax: _d_rough_true(ax, dates, pipes, lab, False),
          lambda ax: _d_rough_true(ax, dates, pipes, lab, True)])
    save("failure_counts", [lambda ax: _d_failures(ax, summ, pipes, lab)])
    save("failure_percentage", [lambda ax: _d_fail_pct(ax, summ, pipes, lab)])
    return saved


# ──────────────── separate stats-table image ────────────────
def plot_stats_table_image(stats, label, outpath):
    """Render the per-pipeline stats table (true reduction, roughness, run-time, iters-to-converge)
    as its OWN image (it used to live inside the compare figure)."""
    cols = [("pipeline", "pipeline", "{}"), ("n", "n", "{:d}"), ("fail", "#fail", "{:d}"),
            ("fail_pct", "fail%", "{:.1f}"),
            ("true_mean", "mean", "{:.2f}"), ("true_med", "med", "{:.2f}"), ("true_std", "std", "{:.2f}"),
            ("rough_mean", "mean", "{:.4f}"), ("rough_med", "med", "{:.4f}"), ("rough_std", "std", "{:.4f}"),
            ("rt_mean", "mean", "{:.2f}"), ("rt_med", "med", "{:.2f}"), ("rt_std", "std", "{:.2f}"),
            ("conv_mean", "mean", "{:.1f}"), ("conv_med", "med", "{:.1f}"), ("conv_std", "std", "{:.1f}")]
    header = [h for _, h, _ in cols]
    cell_text = [[(str(r[k]) if k == "pipeline" else fmt.format(r[k])) for k, _, fmt in cols]
                 for _, r in stats.iterrows()]
    fig, ax = plt.subplots(figsize=(20, 0.55 * len(stats) + 2.2))
    ax.axis("off")
    ax.set_title(f"{_sep(label)}per-pipeline statistics  (n={int(stats['n'].iloc[0])} each)\n"
                 "groups: true reduction % | roughness sqrt(S_final) | run-time (s) | iters to converge",
                 fontweight="bold", fontsize=12)
    tbl = ax.table(cellText=cell_text, colLabels=header, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.6)
    for i in range(len(stats)):              # tint the pipeline-name cell with its palette color
        try:
            tbl[(i + 1, 0)].set_facecolor(PALETTE[i]); tbl[(i + 1, 0)].set_alpha(0.30)
        except Exception:
            pass
    _save(fig, outpath, dpi=130, bbox_inches="tight"); plt.close(fig)
    return outpath


# ──────────────── run_log convergence & feasibility figure ────────────────
def plot_convergence(dates, summ, label, outpath):
    """From run_log trajectories: (1) feasibility track = fraction of runs feasible at each outer
    iteration; (2) boxen of iterations-to-converge; (3) boxen of fraction-of-iters-feasible."""
    import seaborn as sns
    pipes = list(summ["pipeline"]); order = [_short(p) for p in pipes]
    pal = {l: PALETTE[i] for i, l in enumerate(order)}
    d2 = dates.copy()
    d2["plabel"] = pd.Categorical([_short(p) for p in d2["pipeline"]], categories=order, ordered=True)

    fig, ax = plt.subplots(1, 3, figsize=(23, 6.5))
    fig.suptitle(f"{_sep(label)}convergence & feasibility (from run_log)", fontsize=14, fontweight="bold")

    for j, p in enumerate(pipes):
        tracks = [t for t in dates[dates.pipeline == p]["feas_track"] if t is not None]
        if tracks:
            m = np.nanmean(np.vstack(tracks), axis=0)
            ax[0].plot(np.arange(1, len(m) + 1), m, color=PALETTE[j], label=order[j])
    ax[0].set_title("feasibility track: fraction of runs feasible vs outer iteration")
    ax[0].set_xlabel("outer iteration"); ax[0].set_ylabel("fraction of runs feasible")
    ax[0].legend(fontsize=7)

    sns.boxenplot(data=d2, x="plabel", y="iters_to_conv", order=order, hue="plabel",
                  palette=pal, legend=False, ax=ax[1])
    ax[1].set_title("iterations to converge  (first iter within 1% of final J)")
    ax[1].set_xlabel(""); ax[1].set_ylabel("iterations to converge")
    ax[1].set_xticklabels(order, rotation=25, ha="right", fontsize=8)

    sns.boxenplot(data=d2, x="plabel", y="frac_feas", order=order, hue="plabel",
                  palette=pal, legend=False, ax=ax[2])
    ax[2].set_title("fraction of outer iterations feasible")
    ax[2].set_xlabel(""); ax[2].set_ylabel("fraction feasible")
    ax[2].set_xticklabels(order, rotation=25, ha="right", fontsize=8)

    fig.tight_layout(rect=[0, 0, 1, 0.96]); _save(fig, outpath, dpi=120); plt.close(fig)
    return outpath


# ──────────────── compare_2: per-metric facet sets (plain + grey-background overlay) ────────────────
_FACET_METRICS = [
    dict(name="roughness_vs_index", x="__xi__", y="sqrt_S_final", src="d", clip=False,
         xl="data sample index", yl="sqrt(S_final)"),
    dict(name="roughness_vs_tenor", x="T", y="sqrt_S_final", src="d", clip=False,
         xl="T (years)", yl="sqrt(S_final)"),
    dict(name="S_final_vs_index", x="__xi__", y="sqrt_S_final", src="d", clip=False,
         xl="data sample index (sorted by expiry, start date)", yl="sqrt(S_final)"),
    dict(name="S_final_vs_tenor", x="T", y="sqrt_S_final", src="d", clip=False,
         xl="time to expiry  T (years)", yl="sqrt(S_final)"),
    dict(name="runtime_vs_index", x="__xi__", y="winner_runtime", src="d", clip=False,
         xl="data sample index", yl="seconds"),
    dict(name="runtime_vs_tenor", x="T", y="winner_runtime", src="d", clip=False,
         xl="T (years)", yl="seconds"),
    dict(name="true_vs_index", x="__xi__", y="winner_true", src="d", clip=True,
         xl="data sample index", yl="true reduction %"),
    dict(name="true_vs_tenor", x="T", y="winner_true", src="d", clip=True,
         xl="T (years)", yl="true reduction %"),
    dict(name="roughness_vs_true", x="winner_true", y="sqrt_S_final", src="d", clip=False,
         xl="true reduction %", yl="sqrt(S_final)"),
    dict(name="mean_true_by_expiry", x="__er__", y="winner_true", src="me", clip=False,
         xl="expiry index", yl="mean true reduction %"),
]


def _facet_sources(dates):
    # build the per-point frame 'd' (with index column) and the mean-by-expiry frame 'me'
    d = dates.copy()
    keys = sorted(set(zip(d["expiry"], d["start_date"])))
    kidx = {k: i for i, k in enumerate(keys)}
    d["__xi__"] = [kidx[(e, s)] for e, s in zip(d["expiry"], d["start_date"])]
    me = d.groupby(["pipeline", "expiry"], observed=True)["winner_true"].mean().reset_index()
    erank = {e: i for i, e in enumerate(sorted(d["expiry"].unique()))}
    me["__er__"] = me["expiry"].map(erank)
    return d, me


def _facet_limits(full_x, full_y, clip):
    xlo, xhi = np.nanmin(full_x), np.nanmax(full_x)
    if clip:
        ylo, yhi = min(-20, np.nanmin(_clip_lo(full_y))), 102
    else:
        ylo, yhi = min(0.0, np.nanmin(full_y)), float(np.nanpercentile(full_y, 98)) * 1.12
    return (xlo, xhi), (ylo, yhi)


def _draw_facet_row(axes, m, src, pipes, grey):
    fx = src[m["x"]].values
    fy = _clip_lo(src[m["y"]]) if m["clip"] else src[m["y"]].values
    xlim, ylim = _facet_limits(fx, fy, m["clip"])
    for k, (ax, p) in enumerate(zip(axes, pipes)):
        if grey:
            ax.scatter(fx, fy, s=5, color="#d9d9d9", alpha=0.25, linewidths=0, rasterized=True)
        sub = src[src["pipeline"] == p]
        sy = _clip_lo(sub[m["y"]]) if m["clip"] else sub[m["y"]].values
        ax.scatter(sub[m["x"]].values, sy, s=9, color=PALETTE[k], alpha=0.75, linewidths=0, rasterized=True)
        ax.set_xlim(*xlim); ax.set_ylim(*ylim)
        ax.set_title(_short(p), fontsize=8); ax.set_xlabel(m["xl"], fontsize=7)
        ax.tick_params(labelsize=6)
        if k == 0:
            tag = "grey bg = all data" if grey else "plain"
            ax.set_ylabel(f"{m['name']}\n[{tag}]\n{m['yl']}", fontsize=7)


def plot_compare_2(dates, summ, label, outpath, panels_dir):
    os.makedirs(panels_dir, exist_ok=True)
    pipes = list(summ["pipeline"]); n = len(pipes)
    d, me = _facet_sources(dates)
    srcmap = {"d": d, "me": me}

    # combined: each metric -> 2 rows (Set A plain, Set B grey-bg), all stacked
    fig = plt.figure(figsize=(3.1 * n, 2.35 * 2 * len(_FACET_METRICS)))
    gs = fig.add_gridspec(2 * len(_FACET_METRICS), n)
    fig.suptitle(f"{_sep(label)}per-metric facets (Set A: plain, Set B: grey full-data background "
                 "+ pipeline overlay)", fontsize=14, fontweight="bold")
    for mi, m in enumerate(_FACET_METRICS):
        src = srcmap[m["src"]]
        axesA = [fig.add_subplot(gs[2 * mi, c]) for c in range(n)]
        axesB = [fig.add_subplot(gs[2 * mi + 1, c]) for c in range(n)]
        _draw_facet_row(axesA, m, src, pipes, grey=False)
        _draw_facet_row(axesB, m, src, pipes, grey=True)
    fig.tight_layout(rect=[0, 0, 1, 0.985]); _save(fig, outpath, dpi=85); plt.close(fig)

    # per-metric panel images (2 rows x n facets each)
    saved = []
    for m in _FACET_METRICS:
        src = srcmap[m["src"]]
        figp, axp = plt.subplots(2, n, figsize=(3.1 * n, 5.4), squeeze=False)
        figp.suptitle(f"{_sep(label)}{m['name']}  (top: plain | bottom: grey full-data bg + overlay)",
                      fontsize=12, fontweight="bold")
        _draw_facet_row(list(axp[0]), m, src, pipes, grey=False)
        _draw_facet_row(list(axp[1]), m, src, pipes, grey=True)
        figp.tight_layout(rect=[0, 0, 1, 0.93])
        fp = os.path.join(panels_dir, f"facet_{m['name']}.{IMG_EXT}")
        _save(figp, fp, dpi=120); plt.close(figp); saved.append(fp)
    return saved


# ──────────────── compare_3: Boxen plots for run-time, roughness, true reduction ────────────────
def plot_compare_3(dates, summ, label, outpath, panels_dir):
    import seaborn as sns
    os.makedirs(panels_dir, exist_ok=True)
    pipes = list(summ["pipeline"]); order = [_short(p) for p in pipes]
    pal = {l: PALETTE[i] for i, l in enumerate(order)}
    d2 = dates.copy()
    d2["plabel"] = pd.Categorical([_short(p) for p in d2["pipeline"]], categories=order, ordered=True)
    specs = [("winner_runtime", "run-time per date (s)", False),
             ("sqrt_S_final", "roughness sqrt(S_final)", False),
             ("winner_true", "true reduction %", True)]

    def _draw_box(ax, col, title, clip):
        dd = d2.copy()
        dd["__y__"] = _clip_lo(dd[col]) if clip else dd[col].values
        sns.boxplot(data=dd, x="plabel", y="__y__", order=order, hue="plabel",
                    palette=pal, legend=False, fliersize=2, ax=ax)
        ax.set_title(f"{title}"); ax.set_xlabel(""); ax.set_ylabel(title)
        ax.set_xticklabels(order, rotation=25, ha="right", fontsize=8)
        if col == "winner_true":
            ax.set_ylim(min(-20, float(_clip_lo(dd[col]).min())), 101)
            ax.axhline(THRESHOLD, color="k", ls="--", lw=1)
        else:
            ax.set_ylim(0, float(np.nanpercentile(dd[col], 98)) * 1.15)

    fig, axes = plt.subplots(1, 3, figsize=(24, 7))
    fig.suptitle(f"{_sep(label)}Box plots", fontsize=14, fontweight="bold")
    for ax, (col, title, clip) in zip(axes, specs):
        _draw_box(ax, col, title, clip)
    fig.tight_layout(rect=[0, 0, 1, 0.95]); _save(fig, outpath); plt.close(fig)

    saved = []
    for col, title, clip in specs:
        figp, axp = plt.subplots(figsize=(11, 6.5))
        _draw_box(axp, col, title, clip)
        figp.suptitle(label, fontsize=11)
        figp.tight_layout(rect=[0, 0, 1, 0.95])
        fp = os.path.join(panels_dir, f"box_{col}.{IMG_EXT}")
        _save(figp, fp); plt.close(figp); saved.append(fp)
    return saved

In [6]:
# The seven 2011-2012 pipelines.  KEY IDEA: a retry pipeline ALSO ran its primary mode first,
# so from the same artefacts we can score the primary mode ON ITS OWN (ignore the retry) AND
# the full retry pipeline (keep the better of primary/retry).  sig_only [2] and its +sig_nu retry
# come from the SAME root, giving a precise within-root primary-vs-retry comparison.
#   1. sig_nu                          (primary-only)
#   2. sig_nu + sig_only (retry)
#   3. sig_only [1]                    (retry-free root: testing_2011_2012_signu_vs_sigonly)
#   4. sig_only [2]                    (primary-only view of testing_2011_2022_sigonly)
#   5. sig_only [2] + sig_nu (retry)   (retry view of the SAME root as [2])
#   6. sig_only_LKbar                  (primary-only)
#   7. sig_only_LKbar + sig_nu (retry)
# All seven are evaluated on the SAME 616 samples.
PIPE_ORDER_2011 = ["sig_nu", "sig_nu + sig_only (retry)",
                   "sig_only [1]", "sig_only [2]", "sig_only [2] + sig_nu (retry)",
                   "sig_only_LKbar", "sig_only_LKbar + sig_nu (retry)"]
ROOT_SIGNU      = "testing_2011_2012_signu_vs_sigonly"
ROOT_LKBAR      = "testing_2011_2012_sigonlyLU"
ROOT_SIGONLY_RT = "testing_2011_2022_sigonly"   # sig_only primary + sig_nu retry
LABEL_2011 = ""


def _pipeline_view(runs, name, root_key, how, retry_enabled, threshold=THRESHOLD):
    """Build one per-date pipeline view from mode-run records (build_dates-compatible schema).
    how='primary' -> always take the primary-mode run (ignore any retry);
    how='winner'  -> take the best-true run AND charge run-time for BOTH solves (primary+retry)."""
    out = []
    for (folder, sd), g in runs.groupby(["folder", "start_date"]):
        pm = g["primary_mode"].iloc[0]
        prim_rows = g[g["mode"] == pm]
        prim = prim_rows.iloc[0] if len(prim_rows) else g.sort_values("true_reduce").iloc[-1]
        best = g.sort_values("true_reduce").iloc[-1]
        if how == "primary":
            r, runtime, retried = prim, float(prim["run_time"]), False
        else:
            r, runtime, retried = best, float(g["run_time"].sum()), bool(len(g) > 1)
        ptrue = float(prim["true_reduce"]); wtrue = float(r["true_reduce"])
        out.append(dict(
            record_type="date", root=root_key, pipeline=name, primary_mode=pm,
            retry_enabled=retry_enabled, expiry=int(g["expiry"].iloc[0]), start_date=int(sd),
            S0=float(r["S0"]), T=float(r["T"]),
            S_init=float(np.exp(float(r["J_init"]))),            # squared energy  S_init = exp(J_init)
            # TRUE roughness = sqrt(S) (L2 size of the log(V/C'') jumps, same units as the graph)
            sqrt_S_init=float(np.exp(float(r["J_init"]) / 2.0)),
            n_runs=(1 if how == "primary" else int(len(g))), retried=retried,
            retry_mode=(g[g["mode"] != pm]["mode"].iloc[0] if (g["mode"] != pm).any() else ""),
            primary_true=ptrue, primary_pct=float(prim["pct_improve"]),
            primary_runtime=float(prim["run_time"]),
            winner_mode=str(r["mode"]), winner_true=wtrue, winner_pct=float(r["pct_improve"]),
            S_final=float(r["S_final"]), sqrt_S_final=float(r["sqrt_S_final"]),
            winner_runtime=runtime, winner_n_outer=float(r["n_outer"]),
            total_runtime=float(g["run_time"].sum()),
            # run_log trajectory metrics of the CHOSEN run (primary or winner)
            iters_to_conv=float(r.get("iters_to_conv", np.nan)),
            first_feas=float(r.get("first_feas", np.nan)),
            frac_feas=float(r.get("frac_feas", np.nan)),
            final_viol=float(r.get("final_viol", np.nan)),
            feas_track=r.get("feas_track", None),
            primary_failed=bool((ptrue if how == "winner" else wtrue) < threshold),
            winner_failed=bool(wtrue < threshold),
        ))
    return pd.DataFrame(out)


def build_five_2011_2012(base_dir=BASE_DIR):
    r_signu = collect_runs(os.path.join(base_dir, ROOT_SIGNU), ROOT_SIGNU)
    r_lkbar = collect_runs(os.path.join(base_dir, ROOT_LKBAR), ROOT_LKBAR)
    r_sort  = collect_runs(os.path.join(base_dir, ROOT_SIGONLY_RT), ROOT_SIGONLY_RT)
    signu_pn = r_signu[r_signu["primary_mode"] == "sig_nu"]      # retry folders
    signu_so = r_signu[r_signu["primary_mode"] == "sig_only"]    # genuinely retry-free folders
    parts = [
        _pipeline_view(signu_pn, "sig_nu", ROOT_SIGNU, "primary", False),
        _pipeline_view(signu_pn, "sig_nu + sig_only (retry)", ROOT_SIGNU, "winner", True),
        _pipeline_view(signu_so, "sig_only [1]", ROOT_SIGNU, "primary", False),
        # new root: BOTH its standalone sig_only [2] and the +sig_nu retry view (same samples)
        _pipeline_view(r_sort, "sig_only [2]", ROOT_SIGONLY_RT, "primary", False),
        _pipeline_view(r_sort, "sig_only [2] + sig_nu (retry)", ROOT_SIGONLY_RT, "winner", True),
        _pipeline_view(r_lkbar, "sig_only_LKbar", ROOT_LKBAR, "primary", False),
        _pipeline_view(r_lkbar, "sig_only_LKbar + sig_nu (retry)", ROOT_LKBAR, "winner", True),
    ]
    dates = pd.concat(parts, ignore_index=True)
    dates["pipeline"] = pd.Categorical(dates["pipeline"], categories=PIPE_ORDER_2011, ordered=True)
    return dates.sort_values(["pipeline", "expiry", "start_date"]).reset_index(drop=True)


def evaluate_root(root_key, base_dir=BASE_DIR, eval_dir=EVAL_DIR):
    """Evaluate one testing root: parse -> per-date table -> summary -> 2 images + 1 csv."""
    os.makedirs(eval_dir, exist_ok=True)
    runs  = collect_runs(os.path.join(base_dir, root_key), root_key)
    dates = build_dates(runs, root_key)
    summ  = summarize(dates)

    tag   = root_key.replace("testing_", "")
    label = ROOT_CONFIG[root_key]["label"]
    csv_path   = os.path.join(eval_dir, f"eval_{tag}_data.csv")
    main_path  = os.path.join(eval_dir, f"eval_{tag}_main.{IMG_EXT}")
    extra_path = os.path.join(eval_dir, f"eval_{tag}_extra.{IMG_EXT}")

    # one csv: every per-date record + a per-pipeline summary row (record_type distinguishes)
    pd.concat([dates, summ], ignore_index=True).to_csv(csv_path, index=False)

    if len(summ) > 1:                       # >1 pipeline in this root -> comparison images
        plot_main_compare(dates, summ, label, main_path)
        plot_extra_compare(dates, summ, label, extra_path)
    else:
        plot_main_single(dates, summ, label, main_path)
        plot_extra_single(dates, summ, label, extra_path)

    print(f"[{tag}] {len(dates)} dates, {len(summ)} pipeline(s)")
    print(f"        -> {main_path}\n        -> {extra_path}\n        -> {csv_path}")
    return dict(dates=dates, summary=summ, main=main_path, extra=extra_path, csv=csv_path)


def evaluate_2011_2012(base_dir=BASE_DIR, eval_dir=EVAL_DIR):
    """Combined 7-way evaluation of the 2011-2012 pipelines (sig_nu, sig_nu+sig_only, sig_only[1],
    sig_only[2], sig_only[2]+sig_nu, sig_only_LKbar, sig_only_LKbar+sig_nu) — all on the SAME 616 samples.  Produces
    THREE images (core compare, extra compare, the explicit grid figure) + 1 csv + per-plot panels."""
    os.makedirs(eval_dir, exist_ok=True)
    dates = build_five_2011_2012(base_dir)
    summ = summarize(dates)
    tag = "2011_2012"
    P = lambda n: os.path.join(eval_dir, f"eval_{tag}_{n}")

    # one csv (drop the array-valued feasibility track so the file stays tabular)
    csv_path = P("data.csv")
    pd.concat([dates.drop(columns=["feas_track"]), summ], ignore_index=True).to_csv(csv_path, index=False)

    # core / extra comparisons (unchanged)
    plot_main_compare(dates, summ, LABEL_2011, P(f"main.{IMG_EXT}"))
    plot_extra_compare(dates, summ, LABEL_2011, P(f"extra.{IMG_EXT}"))

    # compare_1 : the explicit grid (stats table removed, CDF zoom 99-100, mean-true scatter) + panels
    c1 = P(f"compare_1.{IMG_EXT}"); c1_panels = P("compare_1_panels")
    plot_compare_grid(dates, summ, LABEL_2011, c1)
    panels1 = plot_compare_panels(dates, summ, LABEL_2011, c1_panels)

    # compare_2 : per-metric facet sets (plain + grey-bg overlay) + per-metric panels
    c2 = P(f"compare_2.{IMG_EXT}"); c2_panels = P("compare_2_panels")
    panels2 = plot_compare_2(dates, summ, LABEL_2011, c2, c2_panels)

    # compare_3 : box plots for run-time, roughness, true reduction + panels
    c3 = P(f"compare_3.{IMG_EXT}"); c3_panels = P("compare_3_panels")
    panels3 = plot_compare_3(dates, summ, LABEL_2011, c3, c3_panels)

    # standalone stats-table image + convergence/feasibility figure (from run_log)
    table_path = P(f"stats_table.{IMG_EXT}"); conv_path = P(f"convergence.{IMG_EXT}")
    stats = pipeline_stats_table(dates)
    plot_stats_table_image(stats, LABEL_2011, table_path)
    plot_convergence(dates, summ, LABEL_2011, conv_path)

    # LaTeX table
    tex_path = P("stats.tex"); latex = stats_latex(stats)
    with open(tex_path, "w") as _f:
        _f.write(latex + "\n")

    print(f"[{tag}] {len(dates)} dates, {len(summ)} pipelines (7-way, same 616 samples)")
    for p in (P(f"main.{IMG_EXT}"), P(f"extra.{IMG_EXT}"), c1, c2, c3, table_path, conv_path, csv_path, tex_path):
        print("        ->", p)
    print(f"        -> panels: {len(panels1)} (compare_1), {len(panels2)} (compare_2), "
          f"{len(panels3)} (compare_3)")
    print("\n--- LaTeX-ready stats table -------------------------------------------------\n")
    print(latex)
    return dict(dates=dates, summary=summ, stats=stats, latex=latex,
                compare_1=c1, compare_2=c2, compare_3=c3,
                stats_table=table_path, convergence=conv_path, csv=csv_path, tex=tex_path)


def evaluate_all(base_dir=BASE_DIR, eval_dir=EVAL_DIR):
    """Run everything: combined 7-way 2011-2012 evaluation + the 2024-2025 pipeline."""
    out = {}
    out["2011_2012"] = evaluate_2011_2012(base_dir, eval_dir)
    out["testing_2024_2025"] = evaluate_root("testing_2024_2025", base_dir, eval_dir)
    print("\nDone. Outputs written to", os.path.abspath(eval_dir))
    return out

## Run

Call `evaluate_all()` to regenerate every image and csv under `evaluation/`.
For just the 2011-2012 seven-way set call `evaluate_2011_2012()`; for a single root
call `evaluate_root("testing_2024_2025")`.


In [7]:
results = evaluate_all()

/var/folders/t5/f551fpw93k53k0ygfthd91040000gn/T/ipykernel_15553/3175639530.py:622: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(order, rotation=25, ha="right", fontsize=8)
/var/folders/t5/f551fpw93k53k0ygfthd91040000gn/T/ipykernel_15553/3175639530.py:622: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(order, rotation=25, ha="right", fontsize=8)
/var/folders/t5/f551fpw93k53k0ygfthd91040000gn/T/ipykernel_15553/3175639530.py:622: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(order, rotation=25, ha="right", fontsize=8)
/var/folders/t5/f551fpw93k53k0ygfthd91040000gn/T/ipykernel_15553/3175639530.py:622: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. 

[2011_2012] 4312 dates, 7 pipelines (7-way, same 616 samples)
        -> evaluation/eval_2011_2012_main.svg
        -> evaluation/eval_2011_2012_extra.svg
        -> evaluation/eval_2011_2012_compare_1.svg
        -> evaluation/eval_2011_2012_compare_2.svg
        -> evaluation/eval_2011_2012_compare_3.svg
        -> evaluation/eval_2011_2012_stats_table.svg
        -> evaluation/eval_2011_2012_convergence.svg
        -> evaluation/eval_2011_2012_data.csv
        -> evaluation/eval_2011_2012_stats.tex
        -> panels: 17 (compare_1), 10 (compare_2), 3 (compare_3)

--- LaTeX-ready stats table -------------------------------------------------

\begin{table}[ht]\centering
\caption{2011--2012 optimization pipelines: per-pipeline statistics (true roughness reduction \%, roughness $\sqrt{S_{\mathrm{final}}}$, run-time in seconds). Requires \usepackage{booktabs}.}
\label{tab:pipeline_stats}
\begin{tabular}{lrrrrrrrrrrrrrrr}
\toprule
 & & & & \multicolumn{3}{c}{true reduction \%} & \multicol

### Headline summary across all pipelines

In [8]:
import pandas as pd
summary_cols = ["root", "pipeline", "n_dates", "n_retries", "n_primary_failed",
                "n_winner_failed", "success_rate", "mean_winner_true",
                "mean_sqrt_S_final", "mean_winner_runtime", "mean_retry_rescue"]
all_summ = pd.concat([r["summary"] for r in results.values()], ignore_index=True)
all_summ[summary_cols].round(3)

,root,pipeline,n_dates,n_retries,n_primary_failed,n_winner_failed,success_rate,mean_winner_true,mean_sqrt_S_final,mean_winner_runtime,mean_retry_rescue
0,testing_2011_2012_signu_vs_sigonly,sig_nu,616,0,37,37,0.940,96.101,0.523,9.115,NaN
1,testing_2011_2012_signu_vs_sigonly,sig_nu + sig_only (retry),616,37,37,5,0.992,98.387,0.374,9.762,38.060
2,testing_2011_2012_signu_vs_sigonly,sig_only [1],616,0,7,7,0.989,98.258,0.345,10.200,NaN
3,testing_2011_2022_sigonly,sig_only [2],616,0,7,7,0.989,98.258,0.345,9.861,NaN
4,testing_2011_2022_sigonly,sig_only [2] + sig_nu (retry),616,7,7,6,0.990,98.581,0.330,9.973,28.480
5,testing_2011_2012_sigonlyLU,sig_only_LKbar,616,0,11,11,0.982,97.692,0.379,10.828,NaN
6,testing_2011_2012_sigonlyLU,sig_only_LKbar + sig_nu (retry),616,11,11,10,0.984,97.885,0.373,11.015,10.804
7,testing_2024_2025,sig_only + sig_nu (retry),210,10,10,7,0.967,96.332,0.958,14.607,801.583


### 2011-2012 per-pipeline statistics (true reduction %, roughness, run-time)

The same numbers shown in `eval_2011_2012_stats_table.svg`. The LaTeX version (printed below,
and saved to `evaluation/eval_2011_2012_stats.tex`) needs `\usepackage{booktabs}`.

In [9]:
stats_2011 = results["2011_2012"]["stats"]
display(stats_2011.round(4))
print(results["2011_2012"]["latex"])

,pipeline,n,fail,fail_pct,true_mean,true_med,true_std,rough_mean,rough_med,rough_std,rt_mean,rt_med,rt_std,conv_mean,conv_med,conv_std
0,sig_nu,616,37,6.0065,96.1011,98.7287,14.6842,0.5228,0.2937,1.2226,9.1147,8.1352,2.6961,46.0812,47.5,26.1235
1,sig_nu+sig_only,616,5,0.8117,98.3871,98.7784,3.9652,0.3745,0.2886,0.3537,9.7624,8.2312,4.1512,45.3977,47.0,25.7431
2,sig_only[1],616,7,1.1364,98.2584,99.0851,8.0749,0.3455,0.2409,0.5593,10.1995,9.1537,3.1556,30.6834,32.0,19.8546
3,sig_only[2],616,7,1.1364,98.2575,99.0854,8.0677,0.3451,0.2433,0.5530,9.8608,8.8821,3.1609,30.6412,32.0,20.2094
4,sig_only[2]+sig_nu,616,6,0.9740,98.5812,99.0854,5.7233,0.3302,0.2433,0.4558,9.9733,8.9074,3.3636,30.2110,31.5,19.6599
5,sig_LKbar,616,11,1.7857,97.6922,99.1451,11.3279,0.3791,0.2218,0.9888,10.8282,10.0696,3.2764,31.7403,33.0,20.2427
6,sig_LKbar+sig_nu,616,10,1.6234,97.8851,99.1451,10.3275,0.3729,0.2218,0.9735,11.0152,10.1133,3.5889,31.8328,33.0,20.2050


\begin{table}[ht]\centering
\caption{2011--2012 optimization pipelines: per-pipeline statistics (true roughness reduction \%, roughness $\sqrt{S_{\mathrm{final}}}$, run-time in seconds). Requires \usepackage{booktabs}.}
\label{tab:pipeline_stats}
\begin{tabular}{lrrrrrrrrrrrrrrr}
\toprule
 & & & & \multicolumn{3}{c}{true reduction \%} & \multicolumn{3}{c}{roughness $\sqrt{S_{\mathrm{final}}}$} & \multicolumn{3}{c}{run-time (s)} & \multicolumn{3}{c}{iters to converge} \\
\cmidrule(lr){5-7}\cmidrule(lr){8-10}\cmidrule(lr){11-13}\cmidrule(lr){14-16}
Pipeline & $n$ & \#fail & \#fail\% & mean & median & std & mean & median & std & mean & median & std & mean & median & std \\
\midrule
  sig\_nu & 616 & 37 & 6.0 & 96.10 & 98.73 & 14.68 & 0.5228 & 0.2937 & 1.2226 & 9.11 & 8.14 & 2.70 & 46.1 & 47.5 & 26.1 \\
  sig\_nu+sig\_only & 616 & 5 & 0.8 & 98.39 & 98.78 & 3.97 & 0.3745 & 0.2886 & 0.3537 & 9.76 & 8.23 & 4.15 & 45.4 & 47.0 & 25.7 \\
  sig\_only[1] & 616 & 7 & 1.1 & 98.26 & 99.09 & 8.07 & 0.